# ColdSite-DTI — the whole DAVIS analysis on Kaggle (T4 x2)

Runs on **GPU** what the project otherwise runs on a laptop CPU for a working day: the
faithfulness measurements, the UniProt and KLIFS plausibility ladders, the non-kinase
control, the positional and residue-type nulls, the re-scored cold-level accuracy, and
finally the **audit table with Holm correction over the whole family**.

**It trains nothing.** It reads the 48 trained cells (4 models x 4 levels x 3 seeds) from
datasets you attach, and it refuses to start unless every one of them is the cell this
project already verified — each cell's test AUROC is checked against a table baked into
section 5. That is deliberate: an interrupted checkpoint or a stale copy of MolTrans's
invalid seeds would otherwise be analysed silently.

## What to do

1. **Settings** (right panel): Accelerator **GPU T4 x2**, Internet **On**.
2. **Add Input** — the dataset(s) holding the trained cells. All 48 must be reachable:
   - the 36 DAVIS grid cells (DeepDTA, ColdSite-DTI, HyperAttentionDTI) — account 1's
     `grid36_results.zip`, which is ~217 MB and which Kaggle unzips for you;
   - the 12 MolTrans cells — account 2's own training outputs, already datasets on that
     account (**the retrained seeds 2 and 3**, not the first grid's; if you attach the old
     one, section 5 will say so and stop).

   Section 5 searches every attached input, so attaching several is fine, and if two of
   them hold the same cell it uses the verified copy. Running this on **account 2** avoids
   re-uploading the ~3 GB of MolTrans checkpoints.
3. Leave section 1 as it is unless you know you want something different.
4. **Save Version -> Save & Run All (Commit)**.
5. When it finishes, download `analysis_davis.zip` from the Output tab (section 11).

Roughly 2-4 h on two T4s. It self-stops at 11 h so Kaggle cannot kill the commit before
the output is saved; anything unfinished is simply re-run next commit (finished outputs
are skipped).


## 1. Settings — the only cell you should need to edit

In [ ]:
# ============================================================================
# SETTINGS
# ============================================================================

DATASET = 'davis'

# Which models get the per-model reports (faithfulness, ladders, controls, nulls).
# Default = the two published models, whose analysis is what is still missing.
# ColdSite-DTI's own reports were computed on 2026-09-13 and are in the repo's
# results/analysis_davis_policyA/; add it here only to recompute them on GPU.
MODELS = ['hyperattentiondti', 'moltrans']

# The audit table corrects across the whole family at once, so it always covers every
# audited model, whatever MODELS says. DeepDTA has no attention: accuracy anchor only.
AUDIT_MODELS = ['coldsite_dti', 'hyperattentiondti', 'moltrans']
ALL_MODELS = ['deepdta', 'coldsite_dti', 'hyperattentiondti', 'moltrans']

SEEDS = [1, 2, 3]
MAX_PAIRS = 200          # faithfulness pairs per level (the project's setting)
DEADLINE_HOURS = 11      # self-stop, so Kaggle's 12 h kill cannot lose the output

RUN_STAGE1 = True        # faithfulness, UniProt ladder, non-kinase control (per model)
RUN_STAGE2 = True        # KLIFS ladders, positional nulls, re-scored cold accuracy
RUN_STAGE3 = True        # audit (Holm), positive control, summary
SKIP_EXISTING = True     # a finished output is not recomputed (restore-friendly)

INPUT_ROOT = '/kaggle/input'     # every attached dataset is searched

KNOWN = ['deepdta', 'coldsite_dti', 'hyperattentiondti', 'moltrans']
assert DATASET == 'davis', 'the verification table in section 5 is DAVIS-only'
assert MODELS and all(m in KNOWN for m in MODELS), MODELS
assert all(m in KNOWN for m in AUDIT_MODELS), AUDIT_MODELS
assert SEEDS, 'SEEDS must not be empty'
print(f'{DATASET} | per-model reports: {", ".join(MODELS)} | audit: {", ".join(AUDIT_MODELS)}'
      f' | seeds {SEEDS}')


## 2. The GPUs and the clock

In [ ]:
import time
START = time.time()              # the self-stop is measured from here

import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU.'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)
DEADLINE = START + DEADLINE_HOURS * 3600


def hours_left():
    return (DEADLINE - time.time()) / 3600


print(f'{N_GPU} GPU(s); self-stop in {hours_left():.1f} h')
if N_GPU < 2:
    print('NOTE: one GPU only -- everything still runs, just serially (about twice as long).')


## 3. Clone the repo

The analysis code, the aligned UniProt ground truth, the KLIFS pocket definitions and the
non-kinase panel all live in the repository, so nothing here is fetched by hand.

In [ ]:
import os

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt

RESULTS = f'{WORK}/results'                        # the trained cells land here
OUT = f'{WORK}/analysis_{DATASET}_policyA'         # UniProt analysis outputs
OUT_KLIFS = f'{OUT}_klifs'                         # KLIFS ladders (same file names)
for d in (RESULTS, OUT, OUT_KLIFS):
    os.makedirs(d, exist_ok=True)

# The pieces this notebook cannot run without, all of them committed files.
NEEDED = ['src/evaluation/run_all.py', 'src/evaluation/positional_control.py',
          'src/evaluation/clean_accuracy.py', 'src/evaluation/exclusions.py',
          f'data/{DATASET}_ground_truth_sites.json', f'data/{DATASET}_klifs_pocket_sites.json',
          'data/nonkinase_ground_truth_sites.json', 'data/processed/nonkinase_panel.csv']
missing = [p for p in NEEDED if not os.path.exists(p)]
assert not missing, ('this checkout is missing ' + ', '.join(missing) +
                     ' -- push the commit that adds them to origin/main, then re-run this cell')
print()
!git log --oneline -1
print('cells  ->', RESULTS)
print('outputs->', OUT, 'and', OUT_KLIFS)


## 4. The DAVIS files and the splits

The analysis reads test rows from the split files, so they have to exist here and be the
**same** splits everything was trained on. The row counts are checked against the numbers
recorded on three machines; a mismatch stops the notebook.

`load_data` and `build_splits` each handle both DAVIS and KIBA in one pass, so KIBA's
source files are downloaded as well even though nothing here analyses KIBA.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
# Both loaders below loop over ('davis', 'kiba'), so KIBA's files have to be here too
# even when only DAVIS is analysed -- they are small.
for ds in ('davis', 'kiba'):
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        os.makedirs(os.path.dirname(target), exist_ok=True)
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
        assert os.path.getsize(target) > 1000, f'{target} did not download'
!python -m src.data.load_data 2>&1 | tail -2
!python -m src.data.build_splits 2>&1 | grep -E 'davis|leakage'

import pandas as pd

EXPECTED_SPLITS = {'random': (21039, 3006, 6011), 'cold_drug': (21658, 2652, 5746),
                   'cold_target': (21080, 2992, 5984), 'cold_pair': (15190, 264, 1144)}
problems = []
for split, expected in EXPECTED_SPLITS.items():
    got = tuple(len(pd.read_csv(f'data/splits/{DATASET}/{split}/{p}.csv'))
                for p in ('train', 'valid', 'test'))
    print(f'{split:12s} {str(got):26s} {"OK" if got == expected else "MISMATCH"}')
    if got != expected:
        problems.append(f'{split}: expected {expected}, got {got}')
assert not problems, ('these splits are not the ones the cells were trained on:\n  '
                      + '\n  '.join(problems))
print('\nsplits match the trained cells.')


## 5. Collect the trained cells — and verify every single one

Every attached dataset is searched for the 48 `(model, level, seed)` cells. For each cell
the **test AUROC in its results file must match the value this project verified on the
Mac**, to four decimals. That one check catches, without any judgement on your part:

- a checkpoint that was cut off mid-training (its results file is absent, or its number
  differs);
- MolTrans's invalid seeds 2 and 3 from the first grid, which trained as seed 1 (the
  vendored `models.py` reseeds torch on import) and whose AUROCs are seed 1's;
- an older copy of a cell that was later retrained;
- a dataset that is simply the wrong one.

If several attached datasets hold the same cell, the copy whose AUROC matches is the one
used, and its checkpoint is taken from the same folder. Nothing is guessed: if no copy
matches, the notebook stops and names the cell.

In [ ]:
import glob
import json
import shutil

LEVELS = ('random', 'cold_drug', 'cold_target', 'cold_pair')
TOLERANCE = 1e-4          # the file is a copy of a verified run, so this is exact equality

# Test AUROC of each verified cell (merged folder ~/ColdSite-results/davis_binary,
# 2026-09-14, after account 1 v3 and MolTrans's retrained seeds 2-3).
EXPECTED_AUROC = {
    ('deepdta', 'random', 1): 0.931653,
    ('deepdta', 'random', 2): 0.927969,
    ('deepdta', 'random', 3): 0.927453,
    ('deepdta', 'cold_drug', 1): 0.737688,
    ('deepdta', 'cold_drug', 2): 0.650683,
    ('deepdta', 'cold_drug', 3): 0.686223,
    ('deepdta', 'cold_target', 1): 0.903944,
    ('deepdta', 'cold_target', 2): 0.910353,
    ('deepdta', 'cold_target', 3): 0.908054,
    ('deepdta', 'cold_pair', 1): 0.768019,
    ('deepdta', 'cold_pair', 2): 0.712526,
    ('deepdta', 'cold_pair', 3): 0.702702,
    ('coldsite_dti', 'random', 1): 0.925436,
    ('coldsite_dti', 'random', 2): 0.923188,
    ('coldsite_dti', 'random', 3): 0.923381,
    ('coldsite_dti', 'cold_drug', 1): 0.723254,
    ('coldsite_dti', 'cold_drug', 2): 0.712099,
    ('coldsite_dti', 'cold_drug', 3): 0.726471,
    ('coldsite_dti', 'cold_target', 1): 0.850112,
    ('coldsite_dti', 'cold_target', 2): 0.851250,
    ('coldsite_dti', 'cold_target', 3): 0.869614,
    ('coldsite_dti', 'cold_pair', 1): 0.737520,
    ('coldsite_dti', 'cold_pair', 2): 0.556798,
    ('coldsite_dti', 'cold_pair', 3): 0.576517,
    ('hyperattentiondti', 'random', 1): 0.940107,
    ('hyperattentiondti', 'random', 2): 0.939488,
    ('hyperattentiondti', 'random', 3): 0.931396,
    ('hyperattentiondti', 'cold_drug', 1): 0.720117,
    ('hyperattentiondti', 'cold_drug', 2): 0.757170,
    ('hyperattentiondti', 'cold_drug', 3): 0.803226,
    ('hyperattentiondti', 'cold_target', 1): 0.915160,
    ('hyperattentiondti', 'cold_target', 2): 0.916101,
    ('hyperattentiondti', 'cold_target', 3): 0.913219,
    ('hyperattentiondti', 'cold_pair', 1): 0.696443,
    ('hyperattentiondti', 'cold_pair', 2): 0.655194,
    ('hyperattentiondti', 'cold_pair', 3): 0.730135,
    ('moltrans', 'random', 1): 0.921960,
    ('moltrans', 'random', 2): 0.925391,
    ('moltrans', 'random', 3): 0.921102,
    ('moltrans', 'cold_drug', 1): 0.667724,
    ('moltrans', 'cold_drug', 2): 0.680167,
    ('moltrans', 'cold_drug', 3): 0.706277,
    ('moltrans', 'cold_target', 1): 0.867992,
    ('moltrans', 'cold_target', 2): 0.878513,
    ('moltrans', 'cold_target', 3): 0.875572,
    ('moltrans', 'cold_pair', 1): 0.589877,
    ('moltrans', 'cold_pair', 2): 0.567391,
    ('moltrans', 'cold_pair', 3): 0.548271,
}


def names(model, level, seed):
    """(results file, checkpoint file) as the project's checkpoint_naming writes them."""
    suffix = '' if model == 'coldsite_dti' else f'_{model}'
    return (f'{DATASET}_{level}_binary_seed{seed}{suffix}_results.json',
            f'coldsite_dti_{DATASET}_{level}_binary_seed{seed}{suffix}.pt')


found = {}
for path in glob.glob(f'{INPUT_ROOT}/**/*_results.json', recursive=True):
    found.setdefault(os.path.basename(path), []).append(path)
for path in glob.glob(f'{INPUT_ROOT}/**/*.pt', recursive=True):
    found.setdefault(os.path.basename(path), []).append(path)
print(f'{len(found)} distinct file name(s) across the attached inputs\n')

taken, problems = [], []
for (model, level, seed), expected in sorted(EXPECTED_AUROC.items()):
    res_name, ckpt_name = names(model, level, seed)
    chosen = None
    for candidate in found.get(res_name, []):
        try:
            auroc = json.load(open(candidate))['test_metrics']['auroc']
        except Exception as exc:
            problems.append(f'{model} {level} s{seed}: unreadable {candidate} ({exc})')
            continue
        if abs(auroc - expected) <= TOLERANCE:
            chosen = candidate
            break
    if chosen is None:
        seen = []
        for c in found.get(res_name, []):
            try:
                seen.append(f'{os.path.basename(os.path.dirname(c))} '
                            f'{json.load(open(c))["test_metrics"]["auroc"]:.4f}')
            except Exception:
                seen.append(f'{os.path.basename(os.path.dirname(c))} unreadable')
        problems.append(f'{model} {level} s{seed}: expected AUROC {expected:.4f}, '
                        + (f'attached inputs have {"; ".join(seen)}' if seen
                           else 'no results file found'))
        continue
    ckpt = os.path.join(os.path.dirname(chosen), ckpt_name)
    if not os.path.exists(ckpt):
        problems.append(f'{model} {level} s{seed}: results file matches but its checkpoint '
                        f'{ckpt_name} is not beside it')
        continue
    for src in (chosen, ckpt):
        dst = os.path.join(RESULTS, os.path.basename(src))
        if not os.path.exists(dst):
            try:                       # MolTrans's 12 checkpoints are ~240 MB each
                os.symlink(src, dst)
            except OSError:
                shutil.copy2(src, dst)
    taken.append((model, level, seed))

print(f'verified and copied: {len(taken)}/48 cells')
if problems:
    print(f'\nPROBLEMS ({len(problems)}):')
    for p in problems:
        print('  -', p)
assert not problems, (f'{len(problems)} cell(s) are not the verified ones -- see above. '
                      'Attach the dataset holding the 48 verified cells (or detach a stale '
                      'one) and re-run; nothing is analysed until they all match.')

# A checkpoint with no results file beside it is an unfinished cell. run_all refuses to
# analyse one, and so does this: it would score a half-trained model.
orphans = [os.path.basename(p) for p in glob.glob(f'{RESULTS}/*.pt')
           if not any(os.path.basename(p) == names(m, lv, s)[1] for m, lv, s in taken)]
assert not orphans, f'unfinished cell(s) in {RESULTS}: {orphans}'
print('all 48 cells present, each the verified one, no unfinished checkpoints.')


## 6. The runner

Each stage is a list of real command-line calls — the same commands the project runs on a
laptop — spread over the GPUs. Output is streamed with a `[GPU0 ...]` prefix so two
parallel jobs stay readable, a STATUS line prints every 10 minutes, and everything stops at
the deadline with its finished outputs intact.

In [ ]:
import re
import subprocess
import threading

STATUS_EVERY = 600


def py(module, *args):
    return ['python', '-u', '-m', module, *map(str, args)]


def run_parallel(queues, label):
    """queues: {gpu: [(name, cmd, expected_output), ...]}. True if the deadline cut in."""
    queues = {g: q for g, q in queues.items() if q}
    where = {g: {'name': '-', 'done': 0, 'total': len(q)} for g, q in queues.items()}
    state = {'deadline': False, 'failed': []}
    running = {}

    def worker(gpu, jobs):
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONUNBUFFERED': '1'}
        with open(f'{WORK}/{label}_gpu{gpu}.log', 'a') as log:
            for name, cmd, expected in jobs:
                if time.time() > DEADLINE:
                    return
                if SKIP_EXISTING and expected and os.path.exists(expected):
                    print(f'[skip] GPU{gpu} {name} -- {os.path.basename(expected)} exists',
                          flush=True)
                    where[gpu]['done'] += 1
                    continue
                where[gpu]['name'] = name
                began = time.time()
                print(f'\n{"=" * 70}\n[GPU{gpu}] {name}\n{"=" * 70}', flush=True)
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
                running[gpu] = proc
                for line in proc.stdout:
                    if not re.match(r'\s*(test |train )?batch \d+/', line):   # drop batch spam
                        print(f'[GPU{gpu}] {line}', end='', flush=True)
                    log.write(line)
                code_ = proc.wait()
                where[gpu]['done'] += 1
                took = time.time() - began
                if code_ != 0 or (expected and not os.path.exists(expected)):
                    why = f'exited {code_}' if code_ else f'wrote no {os.path.basename(expected)}'
                    print(f'!! [GPU{gpu}] {name} FAILED ({why}, {took:.0f}s)', flush=True)
                    if not state['deadline']:
                        state['failed'].append(name)
                else:
                    print(f'   [GPU{gpu}] {name} done in {took:.0f}s', flush=True)

    threads = [threading.Thread(target=worker, args=(g, q), daemon=True)
               for g, q in queues.items()]
    for t in threads:
        t.start()
    last = 0.0
    while any(t.is_alive() for t in threads):
        if time.time() - last >= STATUS_EVERY:
            last = time.time()
            parts = [f'GPU{g}: {w["name"]} ({w["done"]}/{w["total"]})' for g, w in where.items()]
            print(f'\n=== STATUS {time.strftime("%H:%M")} | ' + ' | '.join(parts)
                  + f' | {hours_left():.1f} h left ===\n', flush=True)
        if time.time() > DEADLINE and not state['deadline']:
            state['deadline'] = True
            print('\n*** deadline: stopping so this commit saves its output. Re-run the '
                  'notebook with the output attached to finish the rest. ***\n', flush=True)
            for proc in list(running.values()):
                if proc.poll() is None:
                    proc.terminate()
        time.sleep(10)
    if state['failed']:
        print('\nFAILED: ' + ', '.join(state['failed']))
    return state['deadline']


def spread(jobs):
    """Deal jobs to the GPUs, heaviest first, so both finish at about the same time."""
    return {g: jobs[g::N_GPU] for g in range(N_GPU)}


COMMON = ['--split-root', 'data/splits', '--checkpoint-dir', RESULTS]
GT = f'data/{DATASET}_ground_truth_sites.json'
GT_KLIFS = f'data/{DATASET}_klifs_pocket_sites.json'
print('runner ready')


## 7. Stage 1 — faithfulness, the UniProt ladder, the non-kinase control

One `run_all` per model, so the two models run side by side on the two GPUs. `run_all`
wires each ladder to its own model's accuracy file — the mistake that is silent if these
commands are typed by hand.

MolTrans is the slow one (biggest model, and its vendored code keeps dropout on at
inference), so it gets a GPU to itself.

In [ ]:
# Both run_all processes append to the same run_all.log in OUT (whole lines, so it is
# readable but interleaved); the per-GPU logs this notebook writes are the tidy ones.
order = [m for m in ('moltrans', 'hyperattentiondti', 'coldsite_dti', 'deepdta')
         if m in MODELS]
jobs = [(f'{m}: faithfulness + ladder + control',
         py('src.evaluation.run_all', '--dataset', DATASET, '--models', m,
            '--seeds', ','.join(map(str, SEEDS)), '--checkpoint-dir', RESULTS,
            '--results-dir', RESULTS, '--split-root', 'data/splits',
            '--ground-truth', GT, '--out-dir', OUT, '--max-pairs', MAX_PAIRS,
            '--steps', 'faithfulness,ladder,control', '--device', 'cuda',
            *([] if SKIP_EXISTING else ['--no-skip-existing'])),
         None)                        # run_all reports its own per-job outcome
        for m in order]

if RUN_STAGE1 and jobs:
    cut = run_parallel(spread(jobs), 'stage1')
    print('\nStage 1:', 'CUT SHORT by the deadline' if cut else 'finished')
else:
    print('Stage 1 skipped by the settings cell.')


## 8. Stage 2 — KLIFS ladders, the positional nulls, re-scored cold accuracy

Three separate questions, all cheap beside stage 1:

- the **KLIFS ladder**: the same explanations scored against the 85-residue ATP pocket
  instead of UniProt's annotated residues (written to a separate folder, because
  `run_ladder` names its files the same way for either ground truth);
- the **positional and residue-type nulls**: is an above-chance precision@k explained by
  where the attention sits, or by which amino acids it likes, rather than by knowing the
  site;
- **clean accuracy**: each cold-level cell re-scored on the targets that are genuinely
  unseen by sequence, which the summary table needs.

In [ ]:
jobs = []
for m in MODELS:
    for s in SEEDS:
        jobs.append((f'{m}: KLIFS ladder seed {s}',
                     py('src.evaluation.run_ladder', '--model', m, '--task', 'binary',
                        '--dataset', DATASET, '--seed', s, *COMMON,
                        '--ground-truth', GT_KLIFS, '--out-dir', OUT_KLIFS,
                        '--device', 'cuda'),
                     os.path.join(OUT_KLIFS, f'ladder_{m}_{DATASET}_seed{s}.json'
                                  if m != 'coldsite_dti' else f'ladder_{DATASET}_seed{s}.json')))
# --tag is appended to the file stem with no separator, so it carries its own '_'
# (that is how results/analysis_davis_policyA/ is named for ColdSite-DTI).
for m in MODELS:
    for tag, gt in (('_policyA', GT), ('_klifs_policyA', GT_KLIFS)):
        jobs.append((f'{m}: positional + residue nulls ({tag.strip("_")})',
                     py('src.evaluation.positional_control', '--model', m,
                        '--dataset', DATASET, '--seeds', ','.join(map(str, SEEDS)),
                        '--checkpoint-dir', RESULTS, '--out-dir', OUT,
                        '--ground-truth', gt, '--tag', tag, '--device', 'cuda'),
                     os.path.join(OUT, f'positional_control_{m}_{DATASET}{tag}.json')))
jobs.append(('re-scored cold-level accuracy (all four models)',
             py('src.evaluation.clean_accuracy', '--dataset', DATASET,
                '--models', ','.join(ALL_MODELS), '--seeds', ','.join(map(str, SEEDS)),
                '--checkpoint-dir', RESULTS, '--results-dir', RESULTS,
                '--out-dir', OUT, '--device', 'cuda'),
             os.path.join(OUT, f'clean_accuracy_{DATASET}.json')))

if RUN_STAGE2:
    cut = run_parallel(spread(jobs), 'stage2')
    print('\nStage 2:', 'CUT SHORT by the deadline' if cut else 'finished')
else:
    print('Stage 2 skipped by the settings cell.')


## 9. Stage 3 — the audit table, the positive control, the summary

The audit is **one** command over the whole family (every audited model, every level, every
seed, plus a uniform-attention control) so that Holm–Bonferroni is applied once, across all
of it — correcting per model would inflate every claim. It runs after stages 1 and 2
because the positive control reads the ladders and the summary reads everything.

This stage is serial by nature, so it takes one GPU; the other idles.

In [ ]:
clean = os.path.join(OUT, f'clean_accuracy_{DATASET}.json')
jobs = [('audit (Holm over the whole family) + positive control + summary',
         py('src.evaluation.run_all', '--dataset', DATASET,
            '--models', ','.join(ALL_MODELS), '--seeds', ','.join(map(str, SEEDS)),
            '--checkpoint-dir', RESULTS, '--results-dir', RESULTS,
            '--split-root', 'data/splits', '--ground-truth', GT, '--out-dir', OUT,
            '--max-pairs', MAX_PAIRS, '--steps', 'audit,positive,summary',
            '--device', 'cuda', *(['--clean-accuracy', clean] if os.path.exists(clean) else []),
            *([] if SKIP_EXISTING else ['--no-skip-existing'])),
         None)]

if RUN_STAGE3:
    cut = run_parallel({0: jobs}, 'stage3')
    print('\nStage 3:', 'CUT SHORT by the deadline' if cut else 'finished')
else:
    print('Stage 3 skipped by the settings cell.')


## 10. What landed

In [ ]:
def show(path, head=40):
    if os.path.exists(path):
        print(f'\n{"-" * 70}\n{path}\n{"-" * 70}')
        print('\n'.join(open(path).read().splitlines()[:head]))
    else:
        print(f'\n(missing: {path})')


expected = {}
for m in MODELS:
    tag = f'{m}_{DATASET}' if m != 'coldsite_dti' else DATASET
    for s in SEEDS:
        expected[f'faithfulness_{tag}_seed{s}.json'] = OUT
        expected[f'ladder_{tag}_seed{s}.json'] = OUT
        expected[f'ladder_{tag}_seed{s}.json (KLIFS)'] = OUT_KLIFS
        expected[f'control_{m}_{DATASET}_seed{s}_noions.json'] = OUT
    for t in ('_policyA', '_klifs_policyA'):
        expected[f'positional_control_{m}_{DATASET}{t}.json'] = OUT
expected[f'audit_{DATASET}_binary.json'] = OUT
expected[f'positive_control_{DATASET}.json'] = OUT
expected[f'clean_accuracy_{DATASET}.json'] = OUT
expected[f'analysis_summary_{DATASET}.md'] = OUT

missing = []
for name, folder in expected.items():
    real = name.replace(' (KLIFS)', '')
    if not os.path.exists(os.path.join(folder, real)):
        missing.append(name)
print(f'{len(expected) - len(missing)}/{len(expected)} expected outputs present')
if missing:
    print('MISSING (re-run this notebook with this output attached to finish them):')
    for name in missing:
        print('  -', name)

show(os.path.join(OUT, f'analysis_summary_{DATASET}.md'), 80)
show(os.path.join(OUT, f'audit_{DATASET}_binary.md'), 40)


## 11. Take the results with you

One zip of both output folders (they are small — reports, JSON and figures, no
checkpoints). Download it from the **Output** tab, and if a second commit is needed,
attach it as a dataset: section 5 will not re-copy the cells and every finished output is
skipped.

In [ ]:
ZIP = f'{WORK}/analysis_{DATASET}.zip'
FOLDERS = f'{os.path.basename(OUT)} {os.path.basename(OUT_KLIFS)}'
!rm -f {ZIP}
!cd {WORK} && zip -qr {ZIP} {FOLDERS} && zip -qj {ZIP} stage1_gpu0.log stage1_gpu1.log stage2_gpu0.log stage2_gpu1.log stage3_gpu0.log 2>/dev/null
print(f'{ZIP}  {os.path.getsize(ZIP)/1e6:.1f} MB')
!unzip -l {ZIP} | tail -3
print(f'\nelapsed {(time.time() - START)/3600:.1f} h of the {DEADLINE_HOURS} h budget')
